In [4]:
import joblib
import pandas as pd
from pathlib import Path

hiper = pd.read_excel("dados/refined/tabela_melhores_hiperparametros.xlsx")
metricas = pd.read_excel("dados/refined/resultado_modelos_lstm_3.xlsx")

hiper = hiper[hiper['ticker'].isin(metricas["Ticker"].unique())]

features = [
    'close', 'high', 'low', 'open', 'volume',
    'close_dolar', 'close_ibovespa', 'close_sp_500', 'selic', 'ipca',
    'ma20', 'ma50', 'bb_upper', 'bb_lower', 'rsi_wilder', 'macd',
    'macd_signal', 'weekday_sin', 'weekday_cos', 'month_sin', 'month_cos'
]

Path("models").mkdir(exist_ok=True)

for ticker in hiper["ticker"]:

    hp = hiper.loc[
        hiper["ticker"] == ticker
    ].iloc[0].to_dict()

    mt = metricas.loc[
        metricas["Ticker"] == ticker
    ].iloc[0].to_dict()

    metadata = {
        "ticker": ticker,
        "features": features,
        "window_size": int(hp["window_size"]),
        "batch_size": int(hp["batch_size"]),
        "units_1": int(hp["units_1"]),
        "units_2": int(hp["units_2"]),
        "dropout": float(hp["dropout"]),
        "learning_rate": float(hp["learning_rate"]),
        "optimizer": hp["optimizer"],
        "l2": float(hp["l2"]),
        "dense_units": int(hp["dense_units"]),
        "n_dense": int(hp["n_dense"]),
        "metrics": {
            "mae": float(mt["MAE"]),
            "rmse": float(mt["RMSE"]),
            "mape": float(mt["MAPE (%)"]),
            "smape": float(mt["sMAPE (%)"]),
            "r2": float(mt["R2"]),
            "mae_naive": float(mt["MAE Naive"]),
            "rmse_naive": float(mt["RMSE Naive"]),
            "mape_naive": float(mt["MAPE Naive (%)"]),
            "skill_rmse": float(mt["Skill RMSE"])
        }
    }

    joblib.dump(
        metadata,
        f"models/metadata_{ticker}.pkl"
    )